In [ ]:
import logging

from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.magic_state_cultivation import MagicStateCultivation
from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from library.surface_code.patch import SurfaceCodePatch

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 7
SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3

qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (1, 1))
circuitry = Circuitry(qubits, clifford=True, opacity=0.25)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(1, 1)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(circuitry, rnd)
msc.append_cultivation(circuitry)
msc.append_teleportation(circuitry, TELEPORT_ROUNDS)
msc.append_expansion(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)

circuitry.detectors_report()

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_EXPANDED",
    msc.target.logical(Pauli.Y, offset=TARGET_DISTANCE-5),
    *[
        "JCT0:Z0",
        "JCT0:Z1",
        "JCT0:Z2",
        "STN:TPT0:XB",
        "STN:TPT1:XB",
        "STN:TPT2:XB",
        "STN:DST:X1",
        "STN:DST:X5",
        "STN:DST:X6",
    ],
)

display(Markdown(f"[Open in Crumble]({circuitry.to_crumble_url()})"))